In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.io import mmread
from pathlib import Path
import gc

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# UPDATE THIS TO YOUR ACTUAL BASE DIRECTORY
base_dir = Path('.')

# ============================================================================
# STRICT DIRECTORY DISCOVERY
# ============================================================================
study_dirs = [d for d in base_dir.iterdir() if d.is_dir()]
target_dirs = []
for study in study_dirs:
    for subfolder in ['for_use', 'original']:
        expected_path = study / subfolder
        if expected_path.is_dir():
            target_dirs.append(expected_path)

target_dirs = sorted(target_dirs)

print('\n' + '=' * 70)
print('  Data Loading to AnnData (Subset Mode - Bulletproof)')
print('=' * 70)
print(f'Base directory : {base_dir}')
print(f'Datasets found : {len(target_dirs)}\n')


In [ ]:

# ============================================================================
# PROCESSING LOOP
# ============================================================================
for target_dir in target_dirs:

    study_name = target_dir.parent.name
    folder_type = target_dir.name
    sim_name = f"{study_name}_{folder_type}"

    print('=' * 70)
    print(f'Processing: {sim_name}')
    
    subset_file = target_dir / f"{sim_name}_subset.h5ad"
    
    # ------------------------------------------------------------------------
    # BEGIN TRY-EXCEPT BLOCK
    # ------------------------------------------------------------------------
    try:
        # --- 1. VERIFY FILES ------------------------------------------------
        counts_file = target_dir / 'counts.mtx'
        genes_file = target_dir / 'genes.tsv'
        barcodes_file = target_dir / 'barcodes.tsv'
        meta_file = target_dir / 'metadata.tsv'

        if not all(f.exists() for f in [counts_file, genes_file, barcodes_file, meta_file]):
            raise FileNotFoundError("Missing one or more required files (.mtx, genes, barcodes, or metadata).")

        # --- 2. LOAD DATA SAFELY --------------------------------------------
        counts = mmread(target_dir / 'counts.mtx').T.tocsr()
        genes = pd.read_csv(target_dir / 'genes.tsv', sep='\t', header=None)[0].values
        cells = pd.read_csv(target_dir / 'barcodes.tsv', sep='\t', header=None)[0].values
        metadata = pd.read_csv(target_dir / 'metadata.tsv', sep='\t', index_col=0)

        # Force the metadata index to be strings so it perfectly aligns with the cells array
        metadata.index = metadata.index.astype(str)

        # Handle Metadata
        if 'cell_type' in metadata.columns:
            metadata['Ground_Truth_Celltype'] = metadata['cell_type']
            
        if 'Ground_Truth_Celltype' in metadata.columns:
            unique_types = sorted(metadata['Ground_Truth_Celltype'].dropna().unique())
            type_to_cluster = {t: i+1 for i, t in enumerate(unique_types)}
            metadata['Ground_Truth_Cluster'] = metadata['Ground_Truth_Celltype'].map(type_to_cluster)

        # Create AnnData object
        adata = ad.AnnData(X=counts, obs=metadata)
        adata.var_names = genes
        adata.obs_names = cells
        adata.obs_names_make_unique()
        adata.var_names_make_unique()

        # Purge unlabeled cells
        adata = adata[~adata.obs['Ground_Truth_Celltype'].isna()].copy()

        # --- 3. PREPROCESSING & FILTERING -----------------------------------
        sc.pp.filter_cells(adata, min_genes=200) #did not apply to all datasets (PBMCBench), but is a common threshold to remove low-quality cells
        sc.pp.filter_genes(adata, min_cells=3)

        if adata.n_obs == 0 or adata.n_vars == 0:
            raise ValueError(f"QC Filtering removed all data! Cells remaining: {adata.n_obs}, Genes remaining: {adata.n_vars}.")

        print(f'  Cells after QC : {adata.n_obs}')
        print(f'  Genes after QC : {adata.n_vars}')

        adata.layers['counts'] = adata.X.copy()
        sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3')
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        adata.layers['log1p'] = adata.X.copy()

        # --- 4. CREATE SUBSETTED VERSION ------------------------------------
        print(f'  → Creating Subsetted version (2000 HVGs)...')
        adata_sub = adata[:, adata.var.highly_variable].copy()
        sc.pp.scale(adata_sub, max_value=10)
        
        n_comps = min(50, adata_sub.n_obs - 1)
        sc.tl.pca(adata_sub, n_comps=n_comps, mask_var=None)

        adata_sub.write(subset_file)

        # --- 5. CLEAN UP ----------------------------------------------------
        del adata, adata_sub, counts, metadata
        gc.collect()
        print(f'  ✓ Saved: {subset_file.name}\n')

    except Exception as e:
        print(f'\n  [!] ERROR processing {sim_name}: {str(e)}\n')

print('=' * 70)
print('  ✓ Pipeline Finished')
print('=' * 70)